# 04 — Explainability (SHAP)

Use the best model (highest ROC-AUC). SHAP beeswarm summary, bar plot of mean |SHAP|, and optional waterfall for one sample.

In [ ]:
import sys
import os
import pandas as pd
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
os.chdir(ROOT)

from preprocess import (
    load_raw_data,
    find_classification_column,
    create_target_and_filter,
    prepare_features_and_target,
    train_test_split_and_scale,
    FEATURE_COLS,
)
from train import train_all_models
from evaluate import evaluate_all_models
from explain import run_shap_for_best_model

In [ ]:
# Preprocess and train (to get best model)
raw = load_raw_data(data_dir=os.path.join(ROOT, 'data', 'raw'))
class_col = find_classification_column(raw)
df = create_target_and_filter(raw, class_col)
feature_cols = [c for c in FEATURE_COLS if c in df.columns]
X, y = prepare_features_and_target(df, feature_cols=feature_cols, drop_pct_missing=0.30)
X_train, X_test, y_train, y_test, scaler = train_test_split_and_scale(
    X, y, test_size=0.2, random_state=42, use_smote=True
)
models, cv_results = train_all_models(X_train, y_train, cv=5)
metrics_df = evaluate_all_models(models, X_test, y_test, cv_results)
best_name = metrics_df.loc[metrics_df['ROC_AUC'].idxmax(), 'Model']
best_model = models[best_name]
print(f"Best model: {best_name}")

In [ ]:
# SHAP: beeswarm (fig7) and bar (fig8)
shap_values, top_feature = run_shap_for_best_model(
    best_model, best_name, X_train, X_test, figures_dir=os.path.join(ROOT, 'figures')
)
print(f"Highest mean |SHAP| feature: {top_feature}")